Vamos a realizar una copia de seguridad en mi servidor personal de PostgreSQL y actualizar el servidor db de SQLITE del proyecto, cuando confirmemos que los archivos han sido guardados satisfactoriamente y con los formatos deseados.
Borraremos todos los archivos innecesarios.
Para ello creamos un .venv con las librerias indispensables.
'pip install psycopg pandas sqlalchemy numpy'
Para ello haremos una funcion que itere sobre las carpetas donde se alojan los .csv.

In [1]:
#Creamos un script para transferir todos los csv del proyecto a la base de datos local del repositorio sqlite3. Cerciorandonos de su funcionamiento.
import pandas as pd
import sqlite3
import os


carpetas_csv = [
    "C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/",
    "C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/",
]
ruta_db_sqlite = "C:/Users/Josue/4GA.Datascience/4GA.DataScience/src/EcoUE.db"


conexion = sqlite3.connect(ruta_db_sqlite)
cursor = conexion.cursor()


for carpeta in carpetas_csv:
    
    for archivo in os.listdir(carpeta):
        if archivo.endswith(".csv"):
            ruta_archivo = os.path.join(carpeta, archivo)
            nombre_tabla = os.path.splitext(archivo)[0]

            try:
                
                df = pd.read_csv(ruta_archivo)

                
                df.to_sql(nombre_tabla, conexion, if_exists="replace", index=False)

                print(f"Archivo {archivo} transferido a la tabla {nombre_tabla}")

            except Exception as e:
                print(f"Error al procesar el archivo {archivo}: {e}")


conexion.close()

Archivo df_cntrytype.csv transferido a la tabla df_cntrytype
Archivo UE128k.csv transferido a la tabla UE128k
Archivo EspIta.csv transferido a la tabla EspIta
Archivo GerHolBelg.csv transferido a la tabla GerHolBelg
Archivo HungRumEslo.csv transferido a la tabla HungRumEslo
Archivo UEmedG1.csv transferido a la tabla UEmedG1
Archivo UEporG2.csv transferido a la tabla UEporG2
Archivo UEricG0.csv transferido a la tabla UEricG0


Ahora lo mismo para almacenar los csv como tablas en nuestro servidor propio, tendremos que tener cuidado con el archivo configsql.cfg deberemos tenerlo apuntado en el gitignore para no revelar el archivo en el repositorio
en remoto de github.

In [ ]:
import pandas as pd
import psycopg
import os
import configparser


carpetas_csv = [
    "C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/",
    "C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/",
]


config = configparser.ConfigParser()
config.read('C:/Users/Josue/4GA.Datascience/4GA.DataScience/configsql.cfg')


db_config = dict(config['postgresql'])

try:
    
    conn = psycopg.connect(
        host=db_config['host'],
        dbname=db_config['dbname'],
        user=db_config['user'],
        password=db_config['password'],
        port=db_config['port'],
        # sslmode='require',  
        connect_timeout=10
    )
    cur = conn.cursor()

    
    for carpeta in carpetas_csv:
        
        for archivo in os.listdir(carpeta):
            if archivo.endswith(".csv"):
                ruta_archivo = os.path.join(carpeta, archivo)
                nombre_tabla = os.path.splitext(archivo)[0]

                try:
                    
                    df = pd.read_csv(ruta_archivo)

                    
                    columnas = ", ".join([f'"{col}" TEXT' for col in df.columns])
                    cur.execute(f'CREATE TABLE IF NOT EXISTS "{nombre_tabla}" ({columnas});')

                    # Insertar los datos en la tabla
                    for index, row in df.iterrows():
                        valores = ", ".join([f"'{str(val)}'" for val in row.values])
                        cur.execute(f'INSERT INTO "{nombre_tabla}" VALUES ({valores});')

                    print(f"Archivo {archivo} transferido a la tabla {nombre_tabla} en PostgreSQL")

                except Exception as e:
                    print(f"Error al procesar el archivo {archivo}: {e}")

    
    conn.commit()
    cur.close()
    conn.close()

except psycopg.Error as e:
    print(f"Error connecting to PostgreSQL: {e}")

Archivo df_cntrytype.csv transferido a la tabla df_cntrytype en PostgreSQL
Archivo UE128k.csv transferido a la tabla UE128k en PostgreSQL
Archivo EspIta.csv transferido a la tabla EspIta en PostgreSQL
Archivo GerHolBelg.csv transferido a la tabla GerHolBelg en PostgreSQL
Archivo HungRumEslo.csv transferido a la tabla HungRumEslo en PostgreSQL
Archivo UEmedG1.csv transferido a la tabla UEmedG1 en PostgreSQL
Archivo UEporG2.csv transferido a la tabla UEporG2 en PostgreSQL
Archivo UEricG0.csv transferido a la tabla UEricG0 en PostgreSQL


HEMOS DESACTIVADO LA OPCION DE CONECTARNOS MEDIANTE SSL Y SSH, PUESTO QUE MI SERVIDOR SE ALOJA EN ESTE MISMO 
ORDENADOR Y NO ES NECESARIO. HAREMOS UNA PRUEBA DE CARGA DE TABLAS A DATAFRAMES PAR CONFIRMAR LA CONEXION Y EL BUEN ALMACENAMIENTO DE LOS DATOS.